In [1]:
import pandas as pd
from pathlib import Path

In [2]:
raw_dir = "../RawData"
xlsx_files = [f for f in Path(raw_dir).glob("*.xlsx") if not f.name.startswith("~$")]

# fund name = filename (ตรงกับที่ rename ไว้ตอนโหลดจาก am.kkpfg.com)
# ทุกไฟล์ export จาก am.kkpfg.com มี header เดียวกัน: title/ชื่อกอง/ช่วงวันที่ อยู่แถว 0-2
# แล้วค่อยเป็นหัวตารางจริงที่แถว index 3
dfs = {f.stem: pd.read_excel(f, header=3) for f in xlsx_files}

dfs.keys()

/Users/fulinq/.pyenv/versions/3.12.5/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/fulinq/.pyenv/versions/3.12.5/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/fulinq/.pyenv/versions/3.12.5/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/Users/fulinq/.pyenv/versions/3.12.5/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/fulinq/.pyenv/versions/3.12.5/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/fulinq/.pyenv/versions/3.12.5/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/fulinq/.pyenv/versions/3.12.5/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/fulinq/.p

/Users/fulinq/.pyenv/versions/3.12.5/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/fulinq/.pyenv/versions/3.12.5/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/fulinq/.pyenv/versions/3.12.5/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/fulinq/.pyenv/versions/3.12.5/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/fulinq/.p

dict_keys(['KKP MP', 'KKP INDIA-H', 'KKP DIVIDEND', 'KKP VIETNAM', 'KKP EUROPE-UH', 'KKP ACT FIXED', 'KKP S-PLUS', 'KKP CorePath Extra', 'KKP EWUS500-UH', 'KKP SM CAP', 'KKP SEMICON-H', 'KKP CorePath Balanced', 'KKP EMXCN-UH', 'KKP INCOME-H', 'KKP CASH'])

In [3]:
# วันที่จากเว็บเป็นพุทธศักราชแบบย่อ เช่น '28 ส.ค. 69' -> ต้องแปลงเป็นวันที่จริงเอง
thai_month = {
    "ม.ค.": 1, "ก.พ.": 2, "มี.ค.": 3, "เม.ย.": 4, "พ.ค.": 5, "มิ.ย.": 6,
    "ก.ค.": 7, "ส.ค.": 8, "ก.ย.": 9, "ต.ค.": 10, "พ.ย.": 11, "ธ.ค.": 12,
}

def parse_thai_date(s):
    day, month_th, yy = s.split()
    year_ad = 2500 + int(yy) - 543
    return pd.Timestamp(year=year_ad, month=thai_month[month_th], day=int(day))

In [4]:
fund_names = [i for i in dfs.keys()]
fund_names

['KKP MP',
 'KKP INDIA-H',
 'KKP DIVIDEND',
 'KKP VIETNAM',
 'KKP EUROPE-UH',
 'KKP ACT FIXED',
 'KKP S-PLUS',
 'KKP CorePath Extra',
 'KKP EWUS500-UH',
 'KKP SM CAP',
 'KKP SEMICON-H',
 'KKP CorePath Balanced',
 'KKP EMXCN-UH',
 'KKP INCOME-H',
 'KKP CASH']

In [5]:
nav_series = {
    name: (
        df.dropna(subset=["วันที่", "มูลค่าหน่วยลงทุน"])
          .assign(Date=lambda x: x["วันที่"].map(parse_thai_date))
          .set_index("Date")["มูลค่าหน่วยลงทุน"]
          .sort_index()
    )
    for name, df in dfs.items()
}

# เรียงคอลัมน์ตามวันจัดตั้งกองทุน (เก่าสุดก่อน) เหมือนกับที่ทำใน NAV_Clean.ipynb
first_dates = {name: s.index.min() for name, s in nav_series.items()}
pivot_order = sorted(first_dates, key=first_dates.get)
pivot_order

['KKP MP',
 'KKP DIVIDEND',
 'KKP CorePath Balanced',
 'KKP ACT FIXED',
 'KKP SM CAP',
 'KKP CorePath Extra',
 'KKP SEMICON-H',
 'KKP INCOME-H',
 'KKP S-PLUS',
 'KKP EMXCN-UH',
 'KKP INDIA-H',
 'KKP CASH',
 'KKP EWUS500-UH',
 'KKP VIETNAM',
 'KKP EUROPE-UH']

In [6]:
nav_pivot = pd.concat(nav_series, axis=1)
nav_pivot = nav_pivot[pivot_order]
nav_pivot.sort_index(inplace=True, ascending=False)
nav_pivot

,KKP MP,KKP DIVIDEND,KKP CorePath Balanced,KKP ACT FIXED,KKP SM CAP,KKP CorePath Extra,KKP SEMICON-H,KKP INCOME-H,KKP S-PLUS,KKP EMXCN-UH,KKP INDIA-H,KKP CASH,KKP EWUS500-UH,KKP VIETNAM,KKP EUROPE-UH
Date,,,,,,,,,,,,,,,
2026-08-28,12.9752,14.2669,NaN,12.4779,15.0587,NaN,NaN,NaN,10.9207,NaN,NaN,10.2718,NaN,9.7467,NaN
2026-08-27,12.9747,14.2528,NaN,12.4812,15.1647,NaN,29.4543,10.4944,10.9205,16.4845,NaN,10.2714,11.3120,9.7385,NaN
2026-08-26,12.9746,14.3069,18.2555,12.4825,15.1502,13.2384,28.8982,10.5071,10.9198,16.3059,8.2180,10.2711,11.3200,9.6610,11.1543
2026-08-25,12.9744,14.1797,18.2479,12.4792,14.9919,13.2339,28.8266,10.4904,10.9191,16.3198,8.1930,10.2710,11.3000,9.5356,11.1038
2026-08-24,12.9741,14.2141,18.1946,12.4762,14.9921,13.1809,28.3946,10.4677,10.9195,15.9639,8.1277,10.2712,11.2884,9.5738,11.0627
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2008-09-16,10.0062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-09-15,10.0053,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-09-12,10.0024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
out_dir = Path(".")  # this notebook lives in Optimization/
nav_pivot.to_excel(out_dir / "NAV_merged.xlsx", index=True, header=True)